In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler, random_split
import time
import random
import math
from collections import deque
import numpy as np
import h5py
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import gc

from IPython.display import display, clear_output
import ipywidgets as widgets

import dask.array as da

torch.cuda.is_available()


# barebone dataset class
class MyDatasetDumb(Dataset):
    def __init__(self, data_path):
        self.data_path = data_path

    def __len__(self):
        with h5py.File(self.data_path, "r") as f:
            # opens file (it's basically a dict with keys item_0, item_1, ...) and returns length of dict
            return len(f)

    def __getitem__(self, idx):
        with h5py.File(self.data_path, "r") as f:
            # constructs the key for grabbing sample nr idx (item_idx)
            item = f"item_{idx}"
            # accesses each value of the sample and rebuilds the sample dict for one sample
            sample = {key: np.array(f[item][key]) for key in f[item]}
            return sample  
 
class MyDatasetKeepOpen(Dataset):
    def __init__(self, data_path):
        self.data_path = data_path
        self.file      = None

    def __len__(self):
        if self.file is None:
            # if the file is not opened, just open it temporarily to read the length
            with h5py.File(self.data_path, "r") as f:
                # opens file (it's basically a dict with keys item_0, item_1, ...) and returns length of dict
                return len(f)
        else:
            # it the file is already open, readin the length is trivial
            return len(self.file)

    def __getitem__(self, idx):
        if self. file is None:
            self.file = h5py.File(self.data_path, "r")
            
        # constructs the key for grabbing sample number idx (key = item_idx)
        item = f"item_{idx}"
        # accesses each value of the sample and rebuilds the sample dict for one sample
        sample = {key: np.array(self.file[item][key]) for key in self.file[item]}
        return sample 
    
    def close(self):
        # close the file handle when done with the dataset iterating
        if self.file is not None:
            self.file.close()
            self.file = None

class MyDatasetManaged(Dataset):
    def __init__(self, data_path):
        self.data_path = data_path

    def __enter__(self):
        self.file = h5py.File(self.data_path, "r")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.file.close()

    def __len__(self):
        return len(self.file)

    def __getitem__(self, idx):
        item = f"item_{idx}"
        return {key: np.array(self.file[item][key]) for key in self.file[item]}


data_path = "scene_collections/2024-11-15--21-59-16--X20000.h5" 
dataset   = ParallelDataset(data_path)
loader    = DataLoader(dataset, batch_size=128, shuffle=True)

    
start_time = time.time()
for el in loader:
    end_time = time.time()
    print(end_time - start_time)
    start_time = time.time()
    